In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("spark-practice")
    .master("local[*]")
    .config("spark.driver.memory", "2g")
    .config("spark.driver.maxResultSize", "512m")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(spark.version)
print(spark.sparkContext.uiWebUrl)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/01 14:38:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


3.5.1
http://e6872cb89051:4040


In [2]:
from pathlib import Path
from pyspark.sql import functions as F

base_path = Path("spark_core_data").absolute()
base_uri = base_path.as_uri()
print(base_uri)

customer_count = 10_000
order_count = 120_000
product_count = 1_000
item_count = 300_000
event_count = 200_000

customers = (
    spark.range(customer_count)
    .withColumnRenamed("id", "customer_id")
    .withColumn("customer_name", F.concat(F.lit("customer_"), F.col("customer_id")))
    .withColumn("region", F.expr("array('north','south','east','west','central')[int(customer_id % 5)]"))
    .withColumn("segment", F.expr("array('new','regular','vip')[int(customer_id % 3)]"))
)

products = (
    spark.range(product_count)
    .withColumnRenamed("id", "product_id")
    .withColumn("category", F.expr("array('books','electronics','home','sport','beauty','toys')[int(product_id % 6)]"))
    .withColumn("price", (F.rand(11) * 200 + 5).cast("decimal(10,2)"))
)

orders = (
    spark.range(order_count)
    .withColumnRenamed("id", "order_id")
    .withColumn("customer_id", (F.col("order_id") % customer_count).cast("long"))
    .withColumn("order_date", F.expr("date_add(date'2024-01-01', int(order_id % 180))"))
    .withColumn("status", F.expr("array('created','paid','shipped','cancelled')[int(order_id % 4)]"))
    .withColumn("order_amount", (F.rand(21) * 500 + 20).cast("decimal(10,2)"))
)

order_items = (
    spark.range(item_count)
    .withColumnRenamed("id", "order_item_id")
    .withColumn("order_id", (F.col("order_item_id") % order_count).cast("long"))
    .withColumn("product_id", (F.col("order_item_id") % product_count).cast("long"))
    .withColumn("quantity", (F.col("order_item_id") % 5 + 1).cast("int"))
    .withColumn("item_price", (F.rand(31) * 200 + 5).cast("decimal(10,2)"))
)

events = (
    spark.range(event_count)
    .withColumnRenamed("id", "event_id")
    .withColumn("customer_id", (F.col("event_id") % customer_count).cast("long"))
    .withColumn("event_date", F.expr("date_add(date'2024-01-01', int(event_id % 180))"))
    .withColumn("event_type", F.expr("array('view','click','cart','purchase')[int(event_id % 4)]"))
    .withColumn("skew_key", F.when(F.col("event_id") < event_count * 0.8, F.lit("hot_key")).otherwise(F.concat(F.lit("key_"), (F.col("event_id") % 1000))))
    .withColumn("event_value", (F.rand(41) * 100).cast("double"))
)

tables = {
    "customers": customers.repartition(4),
    "orders": orders.repartition(8),
    "products": products.repartition(2),
    "order_items": order_items.repartition(12),
    "events": events.repartition(8),
}

for name, df in tables.items():
    path = f"{base_uri}/{name}"
    df.write.mode("overwrite").parquet(path)
    print(name, df.count(), path)

file:///materials/seminar_04_spark_core/practice/spark_core_data


customers 10000 file:///materials/seminar_04_spark_core/practice/spark_core_data/customers


orders 120000 file:///materials/seminar_04_spark_core/practice/spark_core_data/orders


products 1000 file:///materials/seminar_04_spark_core/practice/spark_core_data/products


order_items 300000 file:///materials/seminar_04_spark_core/practice/spark_core_data/order_items


events 200000 file:///materials/seminar_04_spark_core/practice/spark_core_data/events


In [3]:
orders = spark.read.parquet(f"{base_uri}/orders")
customers = spark.read.parquet(f"{base_uri}/customers")
events = spark.read.parquet(f"{base_uri}/events")

orders.printSchema()
print("orders partitions:", orders.rdd.getNumPartitions())
print("Spark UI:", spark.sparkContext.uiWebUrl)
orders.groupBy("status").count().show()
events.groupBy("skew_key").count().orderBy(F.desc("count")).show(5)

root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- order_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- order_amount: decimal(10,2) (nullable = true)

orders partitions: 4
Spark UI: http://e6872cb89051:4040
+---------+-----+
|   status|count|
+---------+-----+
|     paid|30000|
|  shipped|30000|
|cancelled|30000|
|  created|30000|
+---------+-----+

+--------+------+
|skew_key| count|
+--------+------+
| hot_key|160000|
| key_414|    40|
| key_260|    40|
| key_160|    40|
| key_974|    40|
+--------+------+
only showing top 5 rows



In [4]:
orders.explain(mode = 'formatted')

== Physical Plan ==
* ColumnarToRow (2)
+- Scan parquet  (1)


(1) Scan parquet 
Output [5]: [order_id#171L, customer_id#172L, order_date#173, status#174, order_amount#175]
Batched: true
Location: InMemoryFileIndex [file:/materials/seminar_04_spark_core/practice/spark_core_data/orders]
ReadSchema: struct<order_id:bigint,customer_id:bigint,order_date:date,status:string,order_amount:decimal(10,2)>

(2) ColumnarToRow [codegen id : 1]
Input [5]: [order_id#171L, customer_id#172L, order_date#173, status#174, order_amount#175]


